Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

###### Version 2026.1

In [1]:
NAME = "Samantha A. Salazar"

# Presenting Uncertainty
## School of Information, University of Michigan

## Week 3: Assignment Overview
### The objectives for this week are for you to:
- learn how to construct hypothetical outcome plots (HOPs) and spaghetti plots for a fit line
- practice making HOPs and spaghetti plots

In [2]:
import time
import altair as alt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from sklearn import linear_model
from sklearn import gaussian_process
import numpy as np

import operator
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from mads.lib.path import assets



# Part 1: Learn to plot HOPs and spaghetti plots for linear regression (12 points)

The following salary dataset describes the relationship between someone's salary and the number of years of experience someone has. In this section, we will construct an animated hypothetical outcome plot (HOP) and a spaghetti plot of a linear regression fit to this dataset.

In [3]:
#load dataset
file = assets.find("Salary_Data.csv")
salary_df = pd.read_csv(file)
salary_df.head()

,YearsExperience,Salary
0,1.1,39343.0
1,1.3,46205.0
2,1.5,37731.0
3,2.0,43525.0
4,2.2,39891.0


## 1.1 Construct the basic building blocks of a HOPs visualization

In order to construct a HOPs visualization, we need the following functions:

1. A function to construct an Altair chart of the data: `get_salary_points_chart()`
2. A function to get one bootstrap sample of the linear regression fit: `get_one_bootstrap_salary_fit()`
3. A function to construct an Altair chart of one linear regression fit line: `get_salary_linear_fit_chart()`

Then we will combine all these functions together to make an animation.

### Question 1.1.1 Plot the data (5 points)

Construct a function, `get_salary_points_chart()`, which plots the data in `salary_df` as a scatterplot. The output should look like this:

![A scatterplot of Years of Experience (x axis) against Salary (y axis)](assignment3_salary_points_chart.png)

In [4]:
def get_salary_points_chart():
    '''
    This function should return an altair plot object that is a scatterplot of
    the salary data, with YearsExperience on the x axis and Salary on the y axis
    '''
    # return an Altair scatterplot of the salary data
    # YearsExperience is shown on the x-axis and Salary is shown on the y-axis

    return alt.Chart(salary_df).mark_circle(color = "black").encode(
        x = alt.X("YearsExperience", title = "Years of Experience"),
        y = alt.Y("Salary", title = "Salary")
    )
    
get_salary_points_chart()

alt.Chart(...)

# Commentary

This chart is the basic observed-data layer for the HOPs and spaghetti plots. The x-axis shows `YearsExperience`, the predictor variable, and the y-axis shows `Salary`, the response variable. The scatterplot lets us see the raw relationship before adding any model uncertainty. In an oral exam, I would explain that the points show a strong positive association: as years of experience increase, salary generally increases. This observed-data layer is important because the uncertainty visualizations should always be interpreted relative to the actual data.

### Question 1.1.2 Bootstrap one linear regression fit (2 points)

We will need a function that returns one bootstrap sample of the regression fit. That is, it resamples the dataset with replacement, then fits a linear regression to the data. Fill in the code below to complete the function:

In [ ]:
def get_one_bootstrap_salary_fit():
    '''
    Returns a sklearn.linear_model.LinearRegression model representing 
    a fit to a bootstrap-resampled version of salary_df
    '''
    
    # resample the data with replacement (replace=True) to a data frame with 
    # the same number of data points (frac=1.0)
    resampled_df = salary_df.sample(frac=1.0, replace=True)

    #fit model to resampled data
    X = resampled_df[['YearsExperience']] #[[ ]] subsets so X remains a DataFrame
    y = resampled_df['Salary']            #y should be an array, so we use [ ]
    
    # insert code below using LinearRegression to return a linear regression model
    # with predictor X and outcome variable y
    # train and return one linear regression model on the bootstrap sample
    salary_reg = LinearRegression()
    salary_reg.fit(X, y)

    return salary_reg

# Commentary

This function creates one bootstrap sample by resampling `salary_df` with replacement. Because the sample is drawn with replacement, some original observations may appear multiple times while others may be omitted. A linear regression model is then fit to this resampled dataset. In an oral exam, I would explain that each bootstrap regression represents one plausible fitted relationship we might have gotten if the data collection process had produced a slightly different sample. This is a way to visualize small world uncertainty: uncertainty in the fitted slope and intercept caused by sampling variability.

In [6]:
np.random.seed(1234)
fit = get_one_bootstrap_salary_fit()
assert np.abs(fit.coef_[0] - 10004) < 0.5, "Bootstrap linear regression: slope coefficient does not match the expected value"
assert np.abs(fit.intercept_ - 21485) < 0.5, "Bootstrap linear regression: intercept does not match the expected value"

We can use this function to get a single sample from the bootstrap sampling distribution of the fit (e.g., its slope and intercept). Each time you run the following cell you should get slightly different values:

In [7]:
salary_reg = get_one_bootstrap_salary_fit()
print("Bootstrapped intercept: ", salary_reg.intercept_)
print("Bootstrapped slope:     ", salary_reg.coef_[0])

Bootstrapped intercept:  23608.862570184552
Bootstrapped slope:      9759.092501075675


### Question 1.1.3 Construct an Altair chart of one regression fit (5 points)

To construct a chart of a fit line or fit curve, we first need a *prediction grid*: a set of x values we want to use to make predictions. This should be in the same form as the input to the regression function (i.e., a DataFrame). 

For this example, we will use evenly-spaced values of `"YearsExperience"`, the x value in our charts. Because it is a linear fit, we strictly speaking only need 2 values, but we will use more (101) because it generalizes better. When you plot non-linear relationships (as we will in Part 2), you need a large number of points in your prediction grid so that the curve is smooth.

In [8]:
# construct a prediction grid for the salary dataset with 101 
# evenly-spaced values from the minimum to maximum number of years of experience
salary_pred_grid = pd.DataFrame({'YearsExperience': np.linspace(
    salary_df['YearsExperience'].min(), 
    salary_df['YearsExperience'].max(), 
    num=101
)})

Complete the `get_salary_linear_fit_chart()` so that it displays a single fit line from the linear regression fit passed in to it. The chart should look like this:

![A line chart of Years of Experience (x axis) against Salary (y axis)](assignment3_salary_line_chart.png)

In [9]:
def get_salary_linear_fit_chart(salary_reg, opacity=0.5):
    '''
    Takes a single linear regression fit (as returned by `get_one_bootstrap_salary_fit()`) and
    returns an Altair chart plotting the fit line
    
    Parameters:
    
    - salary_reg: A regression fit
    - opacity: The opacity of the output line
    '''
    #use the model to predict the mean Salary at each x position
    pred_df = pd.DataFrame({
        'YearsExperience': salary_pred_grid['YearsExperience'],
        'Salary': salary_reg.predict(salary_pred_grid)
    })

    #insert code to return an Altair chart showing the fit line using `pred_df`
    #remember to set the opacity of the line mark to the `opacity` value
    #passed into this function (e.g. `mark_line(opacity=opacity)`)
    # draw the fitted regression line
    return alt.Chart(pred_df).mark_line(
        opacity = opacity, 
        color = "red"
    ).encode(
        x = alt.X("YearsExperience", title = "Years of Experience"),
        y = alt.Y("Salary", title = "Salary")
    )

get_salary_linear_fit_chart(salary_reg)

alt.Chart(...)

# Commentary

This function converts one fitted regression model into a visual line. The prediction grid gives the model a smooth set of x-values, and the model predicts salary for each value of years of experience. Altair then draws those predictions as a line. In an oral exam, I would emphasize that the line is not raw data; it is the model's predicted mean salary at each experience level. The opacity argument matters because later we layer many bootstrapped lines together in a spaghetti plot.

## 1.2 Construct HOPs of the salary data

Now that you have all the pieces, you should be able to put them together to construct a HOPs visualization.

First, run the following code chunk a few times: you should notice that the fit line moves each time you run it.

In [10]:
points_chart = get_salary_points_chart()
salary_reg = get_one_bootstrap_salary_fit()
line_chart = get_salary_linear_fit_chart(salary_reg)
line_chart + points_chart

alt.LayerChart(...)

We will use the `interact()` function to run the above code to generate each frame needed in our HOPs. Run the following code, then press the Play button to start the animation:

In [11]:
def get_one_frame(i):
    '''
    Return one frame in the animation
    '''

    time.sleep(.2)

    # get the point chart
    points_chart = get_salary_points_chart()
    
    # fit one bootstrap regression
    salary_reg = get_one_bootstrap_salary_fit()
    
    # get the line chart
    line_chart = get_salary_linear_fit_chart(salary_reg)
    
    #return the combined points + lines chart
    return line_chart + points_chart

interact(get_one_frame, i = widgets.Play(
    value=0,
    min=0,
    max=100,
    step=1,
    description="Press play",
    disabled=False))

interactive(children=(Play(value=0, description='Press play'), Output()), _dom_classes=('widget-interact',))

<function __main__.get_one_frame(i)>

# Commentary

The HOPs visualization shows one bootstrapped regression line at a time. Each frame is a hypothetical outcome: one possible fitted relationship between experience and salary based on a bootstrap resample of the observed data. In an oral exam, I would explain that HOPs use animation to communicate uncertainty dynamically. Instead of showing a static interval, the viewer sees how much the model fit changes from sample to sample. If the line barely moves, uncertainty is low; if it jumps around substantially, uncertainty is higher.

## 1.3 Construct a spaghetti plot of the salary data

The same functions we used to make the HOPs chart above can be used to make a spaghetti plot as well. This time, we will combine all the line charts together instead of playing them frame-by-frame. First, we make a list containing all the line charts (in the `line_charts` variable), then we use `alt.layer()` to layer all of the line charts together. Finally, we add on the chart of the points:

In [ ]:
B = 50

# get `B` bootstrapped fit line charts
# Note opacity=0.1 sets the line opacity so it is easier to see the overlapping lines. Make
# sure your get_salary_linear_fit_chart() function (defined above) properly uses the opacity argument!
line_charts = [get_salary_linear_fit_chart(get_one_bootstrap_salary_fit(), opacity=0.1) for _ in range(B)]

# combine all the line charts together and layer on the points chart
alt.layer(*line_charts) + get_salary_points_chart()

alt.LayerChart(...)

# Commentary

The spaghetti plot shows many bootstrapped regression lines at once. Each low-opacity red line represents one possible fitted model from one bootstrap sample. Where the lines overlap heavily, the model estimate is more stable; where the lines spread out, uncertainty is greater. In an oral exam, I would compare this to HOPs: HOPs show uncertainty sequentially over time, while spaghetti plots show the distribution of possible fits simultaneously. Spaghetti plots are useful for comparing spread, but too many lines can become visually cluttered.

# Part 2: Spaghetti plots for Polynomial Regression (5 points)

To demonstrate the difference in how you must modify your code to fit a new model, in this section we show how to create spaghetti plots for a polynomial regression. We will follow the same steps as before:

1. A function to construct an Altair chart of the data: `get_poly_points_chart()`
2. A function to get one bootstrap sample of the linear regression fit: `get_one_bootstrap_poly_fit()`
3. A function to construct an Altair chart of one linear regression fit line: `get_poly_fit_chart()`

We will generate a dataset with two variables (`x` and `y`), then draw spaghetti plots of a polynomial fit to the dataset.

## 2.1 Generate dataset

First, generate the dataset:

In [13]:
#prepare dataset
np.random.seed(42)
n = 25

original_x = 5 - 4 * np.random.normal(0, 1, n)
original_y = -2 + 3*original_x - 5*(original_x ** 2) + 7*(original_x ** 3) + np.random.normal(0, 1000, n)

poly_df = pd.DataFrame({'x': original_x, 'y': original_y})

## 2.2 Define helper functions

We'll define the polynomial points chart and draw it:

In [14]:
def get_poly_points_chart():
    '''
    This function should return an altair plot object that is a scatterplot of
    the salary data, with YearsExperience on the x axis and Salary on the y axis
    '''
    return alt.Chart(poly_df).mark_circle(color="black").encode(
        x='x',
        y='y'
    )

get_poly_points_chart()

alt.Chart(...)

Then we define the `get_one_bootstrap_poly_fit()` and `get_poly_fit_chart()` functions so we can draw a single fit:

In [15]:
#prediction grid
poly_pred_grid = pd.DataFrame({
    "x": np.linspace(poly_df['x'].min(), poly_df['x'].max(), num=101)
})

def get_one_bootstrap_poly_fit():
    '''Get one bootstrap sampled polynomial regression fit to the data'''
    #resample the data with replacement (replace=True) to a data frame with 
    #the same number of data points (frac=1.0)
    resampled_df = poly_df.sample(frac=1.0, replace=True)

    #fit model to resampled data
    X = resampled_df[['x']] #[[ ]] subsets so X remains a DataFrame
    y = resampled_df['y']   #y should be an array, so we use [ ]
    
    #x must be transformed into polynomials (e.g. x, x^2, x^3 ... up to the value of `degree`)
    polynomial_features = PolynomialFeatures(degree=2)
    X_poly = polynomial_features.fit_transform(X)
    poly_reg = linear_model.LinearRegression()
    poly_reg.fit(X_poly, y)
    
    return poly_reg

def get_poly_fit_chart(poly_reg, opacity=0.5):
    '''
    Takes a single polynomial regression fit (as returned by `get_one_bootstrap_poly_fit()`) and
    returns an Altair chart plotting the fit curve
    
    Parameters:
    
    - poly_reg: A regression fit
    - opacity: The opacity of the output line
    '''
    #use the model to predict y at each x position
    polynomial_features = PolynomialFeatures(degree=2)
    pred_df = pd.DataFrame({
        'x': poly_pred_grid['x'],
        'y': poly_reg.predict(polynomial_features.fit_transform(poly_pred_grid))
    })

    #return an Altair chart showing the fit line
    return alt.Chart(pred_df).mark_line(
        opacity=opacity,
        color='red'
    ).encode(
        x='x',
        y='y'
    )

poly_reg = get_one_bootstrap_poly_fit()
get_poly_fit_chart(poly_reg)

alt.Chart(...)

# 2.3 Draw spaghetti plot for polynomial regression

### Question 2.3.1 Draw a spaghetti plot for the above polynomial regression (5 points)

Using the helper functions defined above (`get_poly_points_chart()`, `get_one_bootstrap_poly_fit()`, and `get_poly_fit_chart()`), draw a spaghetti plot for the example polynomial regression data. Your output should look something like this:

![Polynomial spaghetti plot fit](assignment3_poly.png)



In [16]:
B = 50

# create B bootstrap polynomial regression fits, each drawn as a low-opacity line
line_charts = [
    get_poly_fit_chart(get_one_bootstrap_poly_fit(), opacity = 0.1)
    for _ in range(B)
]

# layer the bootstrap curves and add the original data points on top
alt.layer(*line_charts) + get_poly_points_chart()

alt.LayerChart(...)

# Commentary

This spaghetti plot applies the same bootstrap uncertainty idea to a polynomial regression model. Instead of fitting a straight line, each bootstrap sample fits a curved relationship using polynomial features. In an oral exam, I would explain that polynomial regression can capture nonlinear patterns, but it can also create more unstable fits, especially near the edges of the data. The spread of the curves shows uncertainty in the estimated nonlinear relationship. This is useful because model uncertainty is not limited to straight-line regression; it also applies to more flexible models.

# Part 3: Diabetes dataset (23 points)

In Part 3, we switch to a new data set that describes diabetes disease progression (a metric that captures the progression of diabetes, where higher scores represent a more advanced case of diabetes) and multiple predictors. Apply what you have learned above to construct hypothetical outcome plots and spaghetti plots for the relationship between diabetes disease progression (`disease_progression`) and the `hdl` variable (high-density lipoproteins -- "good chloestrol" that transports chloestrol to the liver). A visualization of the relationship can be found below:

In [17]:
from sklearn.datasets import load_diabetes
X, y = load_diabetes(return_X_y=True)

#create a dataframe containing predictors (diabetes_X) and the response variable (diabetes_y)

diabetes_X = pd.DataFrame(X, columns=["age","sex","bmi","bp", "tc", "ldl", "hdl","tch", "ltg", "glu"])
diabetes_y = pd.DataFrame(y, columns=["disease_progression"])

#also create a combined data frame with both predictors and response variables
diabetes_df = pd.concat([diabetes_y, diabetes_X], axis=1)

#show the hdl versus disease_progression
alt.Chart(diabetes_df).mark_point().encode(
    x="hdl",
    y="disease_progression"
)

alt.Chart(...)

## 3.1 HOPs and spaghetti plots

Use HOPs and spaghetti plots to visualize a regression model predicting `disease_progression` using `hdl`. You can use any model type you like, including linear regression, polynomial regression, or any other regression model type. You will not be judged by the quality of the model you produce (the data are more noise than signal).

### Question 3.1.1 Define helper functions (10 points)

Define the helper functions you will need, including:

1. A function to construct an Altair chart of the data: `get_diabetes_points_chart()`
2. A function to get one bootstrap sample of the fit: `get_one_bootstrap_diabetes_fit()`
3. A function to construct an Altair chart of one regression fit curve: `get_diabetes_fit_chart()`


In [ ]:
# define your helper functions below. Hint: this is also a good place to define a prediction grid

# define helper functions below.
# these functions separate the workflow into:
# 1. plotting the observed diabetes data,
# 2. fitting one bootstrapped regression model,
# 3. drawing one fitted regression line

# create a prediction grid for HDL values.
# the model needs a smooth sequence of HDL values so that we can draw
# a continuous regression line rather than only predicting at observed points
diabetes_pred_grid = pd.DataFrame({
    "hdl": np.linspace(
        diabetes_df["hdl"].min(),  # start at the smallest observed HDL value
        diabetes_df["hdl"].max(),  # end at the largest observed HDL value
        num = 101                  # use 101 evenly spaced values for a smooth line
    )
})


def get_diabetes_points_chart():
    """
    Return a scatterplot of the observed relationship between HDL
    and diabetes disease progression.
    """
    # each point represents one patient/observation in the diabetes dataset
    # HDL is the predictor variable, and disease progression is the outcome
    return alt.Chart(diabetes_df).mark_circle(
        color = "black",
        opacity = 0.6
    ).encode(
        x = alt.X("hdl:Q", title = "HDL"),
        y = alt.Y("disease_progression:Q", title = "Disease Progression")
    )
    
def get_one_bootstrap_diabetes_fit(i = 0):
    """
    Fit one bootstrapped linear regression model.

    The bootstrap sample is created by sampling rows from diabetes_df
    with replacement. This simulates how the fitted relationship might
    change if we had collected a slightly different sample.
    """
    # resample the diabetes data with replacement
    # frac=1 means the bootstrap sample has the same size as the original data
    # random_state=i makes the result reproducible and lets HOPs generate
    # a different fitted line for each animation frame
    boot_df = diabetes_df.sample(
        frac = 1,
        replace = True,
        random_state = i
    )

    # X_boot must be a DataFrame because sklearn expects a 2D feature matrix
    # y_boot is the response variable we are trying to predict
    X_boot = boot_df[["hdl"]]
    y_boot = boot_df["disease_progression"]

    model = LinearRegression()
    model.fit(X_boot, y_boot)

    return model


def get_diabetes_fit_chart(diabetes_reg, opacity = 0.8):
    """
    Return an Altair line chart showing one fitted regression line
    from a bootstrapped diabetes regression model.
    """
    # copy the HDL prediction grid so we do not accidentally modify
    # the original grid used by other functions
    pred_df = diabetes_pred_grid.copy()
    pred_df["disease_progression"] = diabetes_reg.predict(pred_df[["hdl"]])

    # draw the predicted values as a line.
    # this line represents one possible fitted HDL-disease progression relationship
    return alt.Chart(pred_df).mark_line(
        opacity = opacity,
        color = "red"
    ).encode(
        x = alt.X("hdl:Q", title = "HDL"),
        y = alt.Y("disease_progression:Q", title = "Disease Progression")
    )


# display the observed data with one bootstrapped regression fit layered on top
# the black points are the actual observations
# the red line is one fitted model from one bootstrap sample
get_diabetes_points_chart() + get_diabetes_fit_chart(get_one_bootstrap_diabetes_fit())

alt.LayerChart(...)

# Commentary

These helper functions recreate the HOPs/spaghetti workflow for the diabetes dataset. The scatterplot function shows the observed relationship between HDL and disease progression. The bootstrap function resamples the diabetes data with replacement and fits one linear regression model predicting `disease_progression` from `hdl`. The fit-chart function uses a prediction grid to draw the fitted regression line. In an oral exam, I would explain that this structure separates the visualization into reusable pieces: observed data, one bootstrapped model, and one fitted line. That makes it easy to build both HOPs and spaghetti plots from the same functions.

### Question 3.1.2 Create a spaghetti plot for your model (5 points)

Using the helper functions you created above, visualize a spaghetti plot of your model below.


In [33]:
B = 50

# create B bootstrapped linear regression fits
line_charts = [
    get_diabetes_fit_chart(get_one_bootstrap_diabetes_fit(), opacity = 0.1)
    for _ in range(B)
]

# layer all bootstrapped fit lines, then add the observed data points
alt.layer(*line_charts) + get_diabetes_points_chart()

alt.LayerChart(...)

# Commentary

The diabetes spaghetti plot layers many bootstrapped regression lines over the observed data. Each line represents a possible HDL-disease progression relationship from a different resampled version of the dataset. The lines generally suggest a negative relationship, meaning higher HDL tends to be associated with lower predicted disease progression. However, the lines are not identical, and the points are widely scattered, so HDL alone does not explain disease progression very strongly. In an oral exam, I would say this plot visually represents small world uncertainty in the estimated slope and intercept.

### Question 3.1.3 Create a HOPs chart for your model (5 points)

Using the helper functions you created above, visualize a HOPs chart of your model below.

In [ ]:
def get_diabetes_frame(i):
    # return one frame of the diabetes HOPs animation
    # each frame shows one possible regression line from one bootstrapped sample

    # slow the animation slightly so the viewer can see each hypothetical fit
    time.sleep(0.2)

    # fit one bootstrapped regression model
    # the value of i controls the random_state, so each frame produces
    # a different bootstrap sample and therefore a slightly different line
    diabetes_reg = get_one_bootstrap_diabetes_fit(i)

    # layer the observed diabetes data with the current bootstrapped fit
    # the black points stay fixed because they are the observed data
    # the red line changes from frame to frame because it comes from
    # a different bootstrap sample each time
    return (
        get_diabetes_points_chart() + get_diabetes_fit_chart(diabetes_reg, opacity = 0.9)
    )

# create an interactive HOPs animation
# pressing play cycles through many bootstrap regression lines
# the amount of movement in the red line represents uncertainty in the fitted model
interact(get_diabetes_frame, i = widgets.Play(
    value = 0,                        # starting frame       
    min = 0,                          # first bootstrap sample
    max = 100,                        # last bootstrap sample/frame
    step = 1,                         # move one frame at a time
    description = "Press play", 
    disabled = False
))

interactive(children=(Play(value=0, description='Press play'), Output()), _dom_classes=('widget-interact',))

<function __main__.get_diabetes_frame(i)>

#### *Oral-exam reminder: the points do not change because the observed dataset is fixed; only the fitted line changes because each frame uses a different bootstrap resample.*

# Commentary

The diabetes HOPs chart animates the bootstrapped regression fits one frame at a time. Each frame shows one hypothetical fitted relationship between HDL and disease progression. In an oral exam, I would explain that the animation helps viewers perceive uncertainty by watching the fitted line move across bootstrap samples. The degree of uncertainty is represented visually, not numerically: more movement across frames means more uncertainty in the model fit. In this case, the direction of the relationship appears mostly negative, but the exact strength of the relationship varies.

### Question 3.1.4 Reflect on your model (3 points)

Given the visualizations above, reflect on the model you chose and the uncertainty in the relationship between `hdl` and `disease_progression` in these data. Discuss both small world and large world uncertainty. 

---

### STUDENT ANSWER

The HOPs animation shows a generally negative relationship between HDL and disease progression. The degree of uncertainty is represented visually by the varied fitted slope and position of the regression line across each bootstrapped frame. This indicates that the relationship varies depending on which observations are sampled. However, across frames, the lines appear to follow a similar downward pattern, suggesting a negative relationsip (with an unknown strength of uncertainty). This reflects small world uncertainty because the variation comes from sampling variability within the observed dataset.

There is also large world uncertainty because the disease progression is likely affected by many other variables beyond HDL, like age, BMI, genetics, medication use, etc. Therefore, using HDL as the singular causal predictor of disease progression would erroneously exclude relevant confounding factors and non-linear patterns not included in the model. 

Please remember to submit both the HTML and .ipynb formats of your completed notebook. When generating your HTML, be sure to run your complete code first before downloading as HTML. Please remember to work on your explanations and interpretations!

---

# Commentary

For this reflection, the main point is to distinguish small world uncertainty from large world uncertainty. Small world uncertainty refers to uncertainty inside the model setup: the finite sample, bootstrap resampling, and variation in the estimated regression line. The HOPs and spaghetti plots show this uncertainty directly through the movement and spread of fitted lines. Large world uncertainty refers to broader uncertainty outside the simple model: whether HDL is the right predictor, whether the relationship is linear, whether important variables like BMI, blood pressure, age, or medication use are omitted, and whether the dataset generalizes to the broader diabetes population. In an oral exam, I would emphasize that the visualization helps with sampling/model-fit uncertainty, but it does not prove causation or fully quantify clinical uncertainty.